# A/B test walkthrough: plan, check, analyse, decide

**All data in this notebook is simulated** so the true effect is known and the methods can be checked. Replace the simulation cell with your own experiment data to analyse a real test.

Scenario: a product team wants to know whether a redesigned checkout button raises the conversion rate.

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from abkit import sample_size_per_group, srm_check, two_proportion_test
from abkit.simulate import peeking_false_positive_rate

## 1. Plan before you run
Choose the baseline rate, the smallest lift worth shipping (the minimum detectable effect), the significance level and the power. Decide the sample size **before** looking at results.

In [ ]:
baseline = 0.10   # current conversion rate
mde = 0.02        # smallest absolute lift worth acting on (10% -> 12%)
n_per_group = sample_size_per_group(baseline, mde, alpha=0.05, power=0.8)
print(f"Plan: {n_per_group} users per group ({2*n_per_group} total)")

## 2. Simulate the experiment
We pretend the new design truly lifts conversion from 10% to 12%. Real tests do not tell you the truth, which is why we need statistics.

In [ ]:
rng = np.random.default_rng(7)
control = rng.random(n_per_group) < baseline
treatment = rng.random(n_per_group) < baseline + mde
x_c, x_t = int(control.sum()), int(treatment.sum())
print(f"Control: {x_c}/{n_per_group} converted | Treatment: {x_t}/{n_per_group} converted")

## 3. Check the experiment is valid (sample ratio mismatch)
If the split is not what you designed, something is broken, and the result cannot be trusted.

In [ ]:
chi2, p_srm = srm_check(n_per_group, n_per_group)
print(f"SRM chi2={chi2:.2f}, p={p_srm:.3f}  ->  {'OK' if p_srm >= 0.001 else 'INVESTIGATE'}")

## 4. Analyse
Report the effect with a confidence interval, not just a p-value.

In [ ]:
r = two_proportion_test(x_c, n_per_group, x_t, n_per_group)
print(f"Control {r.p_control:.2%} vs treatment {r.p_treatment:.2%}")
print(f"Absolute lift {r.abs_lift:+.2%}, 95% CI [{r.ci_low:+.2%}, {r.ci_high:+.2%}]")
print(f"Relative lift {r.rel_lift:+.1%}, z={r.z:.2f}, p={r.p_value:.4f}")

## 5. Decide
A decision rule written in advance: ship only if the interval excludes zero **and** guardrail metrics (for example refund rate or page speed) are not worse.

In [ ]:
if r.ci_low > 0:
    print("Statistically significant improvement. Check guardrails, then consider shipping.")
elif r.ci_high < 0:
    print("Statistically significant decline. Do not ship.")
else:
    print("Inconclusive. The interval includes zero.")

## 6. Why you must not stop early when the result looks good
Run an A/A test (no real difference) and stop at the first significant result. The false-positive rate climbs as you check more often.

In [ ]:
for looks in (1, 5, 10, 20):
    rate = peeking_false_positive_rate(2000, looks, sims=3000)
    print(f"{looks:>2} looks -> false-positive rate {rate:.1%}")

## 7. Write the decision memo
Use `docs/DECISION_MEMO_TEMPLATE.md`: hypothesis, design, validity checks, result with interval, guardrails, decision, and what you would do next.